# 00 -- Load & Assemble the Freddie Mac data

**What this notebook does (plain English):** The raw Freddie Mac files have no
column headings and split each loan across two files -- one row of facts at the
loan's start, and one row *per month* of its life (millions of rows). This
notebook attaches the official column names, checks the layout is right, finds
whether and when each loan defaulted, and boils everything down to **one tidy
row per loan**. It then stacks three origination years together: **2007 and
2008** (the financial-crisis "downturn" years) and **2015** (a calm year).

**Headline result:** the crisis vintages default far more often than the calm
one -- about **14% (2007)** and **7% (2008)** versus roughly **2% (2015)**.

In [1]:
import sys, os
ROOT = os.getcwd()
if not os.path.isdir(os.path.join(ROOT, 'src')):
    ROOT = os.path.dirname(ROOT)
os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
import warnings; warnings.filterwarnings('ignore')
print('project root:', ROOT)

project root: D:\Jane\Job Search\Github\bank\github project\freddie mac mortgage


In [2]:
# Read all three vintages, apply the 32-column layout, and collapse the
# monthly performance files down to one row per loan (this is the heavy step).
from src import loaders
df = loaders.load_all_vintages('raw data')
print('assembled loan-level table:', df.shape)

assembled loan-level table: (150000, 56)


In [3]:
# Cache the assembled table so every later notebook loads in seconds.
os.makedirs('data/processed', exist_ok=True)
df.to_parquet('data/processed/loan_level.parquet')
print('cached -> data/processed/loan_level.parquet')

cached -> data/processed/loan_level.parquet


In [4]:
# Build a small data-quality summary: loan counts and default rate per vintage.
dq = df.groupby('vintage_year').agg(
    loans=('loan_sequence_number', 'size'),
    defaults=('ever_default', 'sum'),
    disposed_defaults=('disposed', 'sum'),
    median_credit_score=('credit_score', 'median'),
    median_original_upb=('original_upb', 'median'),
)
dq['default_rate'] = (dq['defaults'] / dq['loans']).round(4)
dq = dq.reset_index()

In [5]:
# Save the one results table for this notebook.
from src.output import save_csv
save_csv(dq, 'output/00_data_quality.csv')
dq

,vintage_year,loans,defaults,disposed_defaults,median_credit_score,median_original_upb,default_rate
0,2007,50000,6870,4479,732.0,160000.0,0.1374
1,2008,50000,3677,2134,753.0,183000.0,0.0735
2,2015,50000,1209,136,760.0,200000.0,0.0242


**Reading the table:** each vintage is a 50,000-loan random sample. The
`default_rate` column is the share of loans that ever hit serious default. The
crisis years dwarf 2015 -- the contrast this whole project is built to show.